In [1]:
import pandas as pd
import sqlite3
from logger_master import get_logger
import json 
import time
import random
from datetime import datetime
import requests
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import fake_useragent
from bs4 import BeautifulSoup
import os

/Users/anastasia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
db_path = os.path.join(os.getcwd(), 'GP_DB.db')

connection = sqlite3.connect(db_path)
cursor = connection.cursor()

In [5]:
data = pd.read_sql_query("SELECT * FROM vacancies", connection)
data.head(5)

,id,hh_vac_id,hh_vac_link,title,experience,employment,schedule,salary_raw,salary_from,salary_to,currency,employer,address_raw,area,skills,published_at
0,1,131963477,https://hh.ru/vacancy/131963477,Senior AI Multiagent Systems Engineer,Опыт более 6 лет,None,None,"400 000 – 700 000₽за месяц,на руки",NaN,NaN,None,Simplenight,Москва,None,None,None
1,2,132530194,https://hh.ru/vacancy/132530194,Middle+ AI-инженер,Опыт 1-3 года,None,None,"до320 000₽за месяц,до вычета налогов",NaN,NaN,None,Точка Банк,Москва,None,[],2026-04-27T18:53:55.340855
2,3,131744414,https://hh.ru/vacancy/131744414,Prompt Engineer / AI‑редактор (нейросети и авт...,Опыт 1-3 года,полная,полный день,"80 000 – 100 000₽за месяц,на руки",80000.0,100000.0,RUR,ДЖЕЙКЕТ РАБОТА,Москва,Москва,[],2026-04-27T18:54:50.686408
3,4,131657596,https://hh.ru/vacancy/131657596,AI Engineer (International AI Brand),Опыт более 6 лет,None,None,"5 000 – 7 000$за месяц,на руки",NaN,NaN,None,MT Lab,Москва,None,None,None
4,5,131873094,https://hh.ru/vacancy/131873094,Senior MLOps & BlockchainOps Engineer (DevOps ...,Опыт более 6 лет,полная,полный день,"300 000 – 600 000₽за месяц,на руки",300000.0,600000.0,RUR,Simplenight,Москва,Москва,[],2026-04-27T18:57:51.653715


In [ ]:
data['raw_page'] = None
data

,id,hh_vac_id,hh_vac_link,title,experience,employment,schedule,salary_raw,salary_from,salary_to,currency,employer,address_raw,area,skills,published_at,raw_page
0,1,131963477,https://hh.ru/vacancy/131963477,Senior AI Multiagent Systems Engineer,Опыт более 6 лет,None,None,"400 000 – 700 000₽за месяц,на руки",NaN,NaN,None,Simplenight,Москва,None,None,None,None
1,2,132530194,https://hh.ru/vacancy/132530194,Middle+ AI-инженер,Опыт 1-3 года,None,None,"до320 000₽за месяц,до вычета налогов",NaN,NaN,None,Точка Банк,Москва,None,[],2026-04-27T18:53:55.340855,None
2,3,131744414,https://hh.ru/vacancy/131744414,Prompt Engineer / AI‑редактор (нейросети и авт...,Опыт 1-3 года,полная,полный день,"80 000 – 100 000₽за месяц,на руки",80000.0,100000.0,RUR,ДЖЕЙКЕТ РАБОТА,Москва,Москва,[],2026-04-27T18:54:50.686408,None
3,4,131657596,https://hh.ru/vacancy/131657596,AI Engineer (International AI Brand),Опыт более 6 лет,None,None,"5 000 – 7 000$за месяц,на руки",NaN,NaN,None,MT Lab,Москва,None,None,None,None
4,5,131873094,https://hh.ru/vacancy/131873094,Senior MLOps & BlockchainOps Engineer (DevOps ...,Опыт более 6 лет,полная,полный день,"300 000 – 600 000₽за месяц,на руки",300000.0,600000.0,RUR,Simplenight,Москва,Москва,[],2026-04-27T18:57:51.653715,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26806,26807,132480631,https://hh.ru/vacancy/132480631,Помощник юриста,Без опыта,None,None,"30 000 – 60 000 ₽ за месяц, на руки",NaN,NaN,None,ИП Рыжова Екатерина Вячеславовна,"Орёл, Наугорское шоссе, 5",None,None,None,None
26807,26808,131274336,https://hh.ru/vacancy/131274336,Стажер в юридический отдел/Помощник юриста,Без опыта,None,None,"от 40 000 ₽ за месяц, на руки",NaN,NaN,None,ООО Бюро арбитражных споров Феникс,Санкт-Петербург,None,None,None,None
26808,26809,132283776,https://hh.ru/vacancy/132283776,Юрист,Опыт 3-6 лет,None,None,"150 000 ₽ за месяц, до вычета налогов",NaN,NaN,None,"Персонал-Exclusive, Первое рекрутинговое агент...",Москва,None,None,None,None
26809,26810,132008277,https://hh.ru/vacancy/132008277,Помощник юриста,Без опыта,None,None,"60 000 – 75 000 ₽ за месяц, на руки",NaN,NaN,None,Юридическая компания Аргумент,Новосибирск,None,None,None,None


In [15]:
for index, row in data.iterrows():
    print(index)
    if row['raw_page']:
        pass
    else:
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
            }

            resp = requests.get(row['hh_vac_link'], headers=headers, timeout=5)

            if resp.status_code == 200:
                text = resp.text
                data.loc[index, 'raw_page'] = resp.text
                print(f'УСПЕХ: {text[:200]}')
            else:
                print(f'ОШИБКА --- {resp.status_code}')

        except Exception as ex:
            print(f'код упал с {ex}')
            
        time.sleep(0.05)

0
1
код упал с HTTPSConnectionPool(host='hh.ru', port=443): Max retries exceeded with url: /vacancy/132530194 (Caused by ConnectTimeoutError(<HTTPSConnection(host='hh.ru', port=443) at 0x177d83050>, 'Connection to hh.ru timed out. (connect timeout=5)'))
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214


KeyboardInterrupt: 

In [16]:
data['raw_page']

0        <!DOCTYPE html>\n<html class="desktop" lang="r...
1                                                     None
2        <!DOCTYPE html>\n<html class="desktop" lang="r...
3        <!DOCTYPE html>\n<html class="desktop" lang="r...
4        <!DOCTYPE html>\n<html class="desktop" lang="r...
                               ...                        
26806                                                 None
26807                                                 None
26808                                                 None
26809                                                 None
26810                                                 None
Name: raw_page, Length: 26811, dtype: object

In [17]:
data.to_excel('batch_v1_size_1000.xlsx')